In [10]:
import pandas as pd
import numpy as np

## Feature Engineering 

### Delivery Time

In [12]:
main_df = pd.read_csv('../data/cleaned_dataset.csv')
main_df.isnull().sum()


order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 141
order_delivered_carrier_date     1961
order_delivered_customer_date    1960
order_estimated_delivery_date       0
order_item_id                     767
product_id                        767
seller_id                         767
shipping_limit_date               767
price                             767
freight_value                     767
product_category_name               0
product_name_length              2370
product_description_length       2370
product_photos_qty               2370
product_weight_g                  785
product_length_cm                 785
product_height_cm                 785
product_width_cm                  785
seller_zip_code_prefix            767
seller_city                       767
seller_state                      767
payment_type_count                  3
used_credit_

In [51]:
main_df['delivery_time_days'] = np.nan

# Correcting DataType
main_df['order_purchase_timestamp'] = pd.to_datetime(
    main_df['order_purchase_timestamp']
)
main_df['order_delivered_customer_date'] = pd.to_datetime(
    main_df['order_delivered_customer_date']
)


mask = (
    (main_df['order_status'] == 'delivered') &
    (main_df['order_delivered_customer_date'].notna())
)

main_df.loc[mask, 'delivery_time_days'] = (
    main_df.loc[mask, 'order_delivered_customer_date'] -
    main_df.loc[mask, 'order_purchase_timestamp']
).dt.days

main_df['delivery_time_days'].head()

0     8.0
1    13.0
2     9.0
3    13.0
4     2.0
Name: delivery_time_days, dtype: float64

### Shipping Duration

In [50]:
main_df['shipping_duration'] = np.nan

# Correcting DataType
main_df['order_delivered_carrier_date'] = pd.to_datetime(
    main_df['order_delivered_carrier_date']
)

mask = (
    (main_df['order_delivered_carrier_date'].notna()) &
    (main_df['order_delivered_customer_date'].notna())
)

main_df.loc[mask, 'shipping_duration'] = (
    main_df.loc[mask, 'order_delivered_customer_date'] -
    main_df.loc[mask, 'order_delivered_carrier_date']
).dt.days

main_df['shipping_duration'].head()

0     6.0
1    12.0
2     9.0
3     9.0
4     1.0
Name: shipping_duration, dtype: float64

### Order Processing Time

In [49]:
main_df.loc[(main_df['order_approved_at'].isnull()) &
            ~(main_df['order_status'] == 'delivered')] # 141 rows

main_df['order_processing_days'] = np.nan

# Correcting DataType
main_df['order_approved_at'] = pd.to_datetime(
    main_df['order_approved_at']
)
main_df['order_purchase_timestamp'] = pd.to_datetime(
    main_df['order_purchase_timestamp']
)

mask = (
    (main_df['order_approved_at'].notna()) &
    (main_df['order_status'] == 'delivered')
)

main_df.loc[mask, 'order_processing_days'] = (
    main_df.loc[mask, 'order_approved_at'] -
    main_df.loc[mask, 'order_purchase_timestamp']
).dt.days

main_df['order_processing_days'].head()


0    0.0
1    1.0
2    0.0
3    0.0
4    0.0
Name: order_processing_days, dtype: float64

### Customer Lifetime Value

In [48]:
main_df['customer_lifetime_value'] = (
    main_df.groupby('customer_unique_id')['total_payment_value']
           .transform('sum')
)
main_df['customer_lifetime_value'].head()

0     82.82
1    141.46
2    179.12
3     72.20
4     28.62
Name: customer_lifetime_value, dtype: float64

### Average Product Price

In [47]:
main_df['Average_product_price'] = (
    main_df.groupby('product_id')['price']
           .transform('mean')
)
main_df['Average_product_price'].head()

0     29.990000
1    119.216038
2    159.233333
3     42.750000
4     21.800000
Name: Average_product_price, dtype: float64

### Customer Purchase Count

In [46]:
main_df['customer_purchase_count'] = main_df.groupby('customer_unique_id')['order_id'].transform('nunique')

main_df['customer_purchase_count'].head()


0    2
1    1
2    1
3    1
4    1
Name: customer_purchase_count, dtype: int64